# Lab 10 - Apple Brand Monitor Data Analysis

This notebook adapts the Lab 10 movie analytics workflow to the `social_media_brand_monitor` project. The cleaned dataset is restricted to Apple mentions, then combined with MySQL and MongoDB results for reshaping, aggregation, pivoting, time-series analysis, and analytical reporting.

## Project-Specific Fields

This submission is for the `social_media_brand_monitor` project focused on Apple mentions, not a movie dataset.

- cleaned input file: `data/processed/cleaned/cleaned_data.csv`
- MySQL table: `apple_article_metrics`
- numeric analysis fields: `rating`, `title_length`, `overview_length`
- grouping fields: `mention_year`, `primary_keyword`
- date field: `mention_date`

In [ ]:
from pathlib import Path
import logging
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.aggregator import add_length_metrics, source_summary, top_n_per_group, yearly_trends
from src.analytics.data_combiner import compare_join_types, merge_on_key, save_join_comparison_chart
from src.analytics.db_connector import create_article_metrics_table, get_mysql_connection, populate_article_metrics, query_article_metrics
from src.analytics.explorer import filter_apple_mentions
from src.analytics.insight_reporter import (
    language_distribution,
    run_all_questions,
    save_insight_summary,
    save_keyword_chart,
    save_language_distribution_chart,
    save_question_report,
    save_source_share_chart,
    save_yearly_volume_chart,
    source_share_summary,
    top_keywords_by_mention_count,
    yearly_release_volume,
)
from src.analytics.mongo_pipeline import build_source_mentions_pipeline, run_pipeline as run_mongo_aggregation
from src.analytics.pivot_builder import add_primary_keyword, build_keyword_year_pivot, build_language_year_crosstab, melt_metrics
from src.analytics.time_series import add_rolling_averages, build_monthly_mentions, parse_mention_dates, resample_mentions, save_rolling_mentions_chart
from src.utils.logger import get_logger

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = get_logger("lab10_data_analysis")

CLEANED_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned" / "cleaned_data.csv"
ANALYTICS_DIR = PROJECT_ROOT / "data" / "processed" / "analytics"
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)


## Load and Prepare the Cleaned Apple Dataset

In [ ]:
cleaned_df = pd.read_csv(CLEANED_CSV_PATH)
cleaned_df = filter_apple_mentions(cleaned_df, keyword="apple")
cleaned_df = parse_mention_dates(cleaned_df)
cleaned_df["rating"] = pd.to_numeric(cleaned_df.get("rating"), errors="coerce")
cleaned_df = add_length_metrics(cleaned_df)
cleaned_df = add_primary_keyword(cleaned_df)
logger.info("Loaded and prepared %s Apple rows for Lab 10 analysis", len(cleaned_df))
cleaned_df.head()

## MySQL Connection and Relational Table

This section fulfills the relational-database requirement: connect to MySQL, create the Apple metrics table, populate it from the cleaned CSV-derived DataFrame, and read it back with `pd.read_sql()`.

In [ ]:
mysql_connection = get_mysql_connection()
create_article_metrics_table(mysql_connection)
populate_article_metrics(mysql_connection, cleaned_df)
mysql_df = query_article_metrics(where_clause="mention_year IS NOT NULL")
mysql_connection.close()
mysql_df.head()

## Combining Data from MySQL and the Cleaned CSV

In [ ]:
metadata_df = cleaned_df[["_id", "title", "source", "document_type", "language", "mention_year", "mention_month", "mention_weekday", "mention_quarter", "overview", "mention_date", "rating", "title_length", "overview_length", "primary_keyword"]].copy()
mysql_join_df = mysql_df.rename(columns={"mention_id": "_id"}).copy()
mysql_join_df = mysql_join_df.rename(columns={column: f"mysql_{column}" for column in mysql_join_df.columns if column != "_id"})
join_comparison = compare_join_types(metadata_df, mysql_join_df, key="_id")
save_join_comparison_chart(join_comparison, ANALYTICS_DIR / "join_type_comparison.png")
join_comparison

**Most appropriate join:** `left` join.

The cleaned Apple dataset is the primary analytical dataset, so every cleaned Apple mention should be preserved even if a relational enrichment row is missing. `left` join keeps the full cleaned dataset and adds MySQL metrics when available. `inner` could silently drop cleaned mentions, while `right` and `outer` are less appropriate because MySQL is a secondary enrichment source here.

In [ ]:
combined_df = merge_on_key(metadata_df, mysql_join_df, key="_id", how="left")
combined_df.head()

## Reshaping Data with `melt()` and `pivot_table()`

In [ ]:
long_metrics = melt_metrics(
    combined_df,
    id_vars=["_id", "title", "source", "primary_keyword", "mention_year"],
    value_vars=["rating", "title_length", "overview_length"],
)
long_metrics.to_csv(ANALYTICS_DIR / "long_metrics.csv", index=False)
long_metrics.head()

In [ ]:
keyword_year_pivot = build_keyword_year_pivot(combined_df)
keyword_year_pivot.to_csv(ANALYTICS_DIR / "pivot_keyword_year.csv")
keyword_year_pivot

The pivot table above is the Apple-domain equivalent of a movie `release_year x primary_genre` summary. Here it shows Apple mention counts broken down by `mention_year` and `primary_keyword`, with `margins=True` to include subtotals.

In [ ]:
language_year_crosstab = build_language_year_crosstab(combined_df)
language_year_crosstab.to_csv(ANALYTICS_DIR / "language_year_crosstab.csv")
language_year_crosstab

## GroupBy Analysis

In [ ]:
source_analysis = source_summary(combined_df)
source_analysis.to_csv(ANALYTICS_DIR / "source_analysis.csv", index=False)
source_analysis.head(10)

The grouped source summary uses named aggregations and multiple functions: `count`, `mean`, `median`, and `sum`.

In [ ]:
yearly_analysis = yearly_trends(combined_df)
yearly_analysis.to_csv(ANALYTICS_DIR / "yearly_trends.csv", index=False)
yearly_analysis

In [ ]:
top_titles_by_keyword = top_n_per_group(combined_df, group_column="primary_keyword", sort_column="overview_length", n=3)
top_titles_by_keyword.to_csv(ANALYTICS_DIR / "top_titles_by_keyword.csv", index=False)
top_titles_by_keyword.head(15)

## Time Series Analysis

In [ ]:
print("Valid mention dates:", combined_df["mention_date"].notna().sum())
print("Missing mention dates:", combined_df["mention_date"].isna().sum())
combined_df[["mention_date", "mention_year", "mention_month", "mention_weekday", "mention_quarter"]].head()

In [ ]:
monthly_mentions = build_monthly_mentions(combined_df)
monthly_mentions = add_rolling_averages(monthly_mentions)
monthly_mentions.to_csv(ANALYTICS_DIR / "monthly_mentions.csv", index=False)
save_rolling_mentions_chart(monthly_mentions, ANALYTICS_DIR / "rolling_mentions.png")
monthly_mentions.head()

In [ ]:
yearly_resampled = resample_mentions(combined_df, frequency="YE")
yearly_resampled.to_csv(ANALYTICS_DIR / "resampled_yearly_mentions.csv", index=False)
yearly_resampled

## MongoDB Aggregation Pipeline

In [ ]:
mongo_pipeline_df = run_mongo_aggregation(build_source_mentions_pipeline())
mongo_pipeline_df.to_csv(ANALYTICS_DIR / "mongo_source_mentions.csv", index=False)
mongo_pipeline_df.head(10)

This pipeline uses `$match`, `$group`, `$sort`, and `$project`, then converts the result into a pandas DataFrame.

## Analytical Questions

In [ ]:
insights = run_all_questions(combined_df)
save_keyword_chart(insights["keyword_summary"], ANALYTICS_DIR / "question1_top_keywords.png")
save_source_share_chart(insights["source_summary"], ANALYTICS_DIR / "question2_top_sources.png")
save_yearly_volume_chart(insights["yearly_summary"], ANALYTICS_DIR / "question3_yearly_volume.png")
save_language_distribution_chart(insights["language_summary"], ANALYTICS_DIR / "question4_language_distribution.png")
save_insight_summary(insights, ANALYTICS_DIR / "insight_summary.txt")
save_question_report(insights, ANALYTICS_DIR / "analytical_questions.txt")
insights.keys()

### Question 1. Which Apple topic appears most often in the cleaned dataset?

In [ ]:
question1 = top_keywords_by_mention_count(combined_df)
question1.head(10)

Quantified finding: the first row of `question1` identifies the top Apple topic and its exact mention count. Chart saved as `data/processed/analytics/question1_top_keywords.png`.

### Question 2. Which source contributes the largest share of Apple coverage?

In [ ]:
question2 = source_share_summary(combined_df)
question2.head(10)

Quantified finding: the first row of `question2` identifies the top source, raw mention count, and percentage share. Chart saved as `data/processed/analytics/question2_top_sources.png`.

### Question 3. In which year does Apple mention volume peak?

In [ ]:
question3 = yearly_release_volume(combined_df)
question3

Quantified finding: the maximum `mention_count` in `question3` identifies the peak year and its volume. Chart saved as `data/processed/analytics/question3_yearly_volume.png`.

### Question 4. What is the dominant language in Apple coverage?

In [ ]:
question4 = language_distribution(combined_df)
question4

Quantified finding: the first row of `question4` identifies the dominant language, its row count, and its percentage share. Chart saved as `data/processed/analytics/question4_language_distribution.png`.

## Pipeline Integration

The same analytics workflow is integrated into `src/pipeline/run_pipeline.py` through `run_analytics_pipeline(cleaned_csv_path)`, so the full project runs with one command:

```bash
python src/pipeline/run_pipeline.py
```